In [2]:
import pandas as pd
from pathlib import Path

# Load Data

In [3]:
root_dir = Path.cwd().parent
data_path = root_dir / "data/raw"

street_list = []
outcomes_list = []
search_list = []

for folder in data_path.iterdir():
    if folder.is_dir():
        street_file = folder / f"{folder.name}-north-yorkshire-street.csv"
        outcomes_file = folder / f"{folder.name}-north-yorkshire-outcomes.csv"
        search_file = folder / f"{folder.name}-north-yorkshire-stop-and-search.csv"

        if street_file.exists():
            df = pd.read_csv(street_file)
            df["source_month"] = folder.name
            street_list.append(df)

        if outcomes_file.exists():
            df = pd.read_csv(outcomes_file)
            df["source_month"] = folder.name
            outcomes_list.append(df)

        if search_file.exists():
            df = pd.read_csv(search_file)
            df["source_month"] = folder.name
            search_list.append(df)

street = pd.concat(street_list, ignore_index = True)
outcomes = pd.concat(outcomes_list, ignore_index = True)
search = pd.concat(search_list, ignore_index = True)

# Street EDA

In [4]:
print("Street Shape:", street.shape)

Street Shape: (101106, 13)


In [5]:
print("Street Info:", street.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101106 entries, 0 to 101105
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Crime ID               75650 non-null   object 
 1   Month                  101106 non-null  object 
 2   Reported by            101106 non-null  object 
 3   Falls within           101106 non-null  object 
 4   Longitude              100499 non-null  float64
 5   Latitude               100499 non-null  float64
 6   Location               101106 non-null  object 
 7   LSOA code              100498 non-null  object 
 8   LSOA name              100498 non-null  object 
 9   Crime type             101106 non-null  object 
 10  Last outcome category  75650 non-null   object 
 11  Context                0 non-null       float64
 12  source_month           101106 non-null  object 
dtypes: float64(3), object(10)
memory usage: 10.0+ MB
Street Info: None


In [6]:
street.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context,source_month
0,c8e8023cef720b5a5921edf441ad06be0b725674807421...,2024-07,North Yorkshire Police,North Yorkshire Police,-2.002397,53.870949,On or near Long Gate,E01010855,Bradford 023B,Burglary,Investigation complete; no suspect identified,NaN,2024-07
1,da5bb18a84998407e3858e8f5ad543905aac6f7dd8d3a1...,2024-07,North Yorkshire Police,North Yorkshire Police,-1.965052,53.936016,On or near Green Lane,E01010638,Bradford 062A,Burglary,Investigation complete; no suspect identified,NaN,2024-07
2,11d31016370b0e2359254d412f7a3a5d8ed07fc77b7c85...,2024-07,North Yorkshire Police,North Yorkshire Police,-1.966311,53.923434,On or near Dennis Lane,E01010638,Bradford 062A,Other theft,Unable to prosecute suspect,NaN,2024-07
3,e67301833184dbde852d19593fa24568711c87260656a4...,2024-07,North Yorkshire Police,North Yorkshire Police,-2.534835,54.119365,On or near Harley Close,E01027558,Craven 001A,Criminal damage and arson,Unable to prosecute suspect,NaN,2024-07
4,902059b04de20f1b0b2aa338f1edd7073af705e166d1a4...,2024-07,North Yorkshire Police,North Yorkshire Police,-2.530959,54.147648,On or near Ireby Road,E01027558,Craven 001A,Public order,Investigation complete; no suspect identified,NaN,2024-07


In [7]:
street.isna().sum().sort_values(ascending = False)

Context                  101106
Last outcome category     25456
Crime ID                  25456
LSOA code                   608
LSOA name                   608
Latitude                    607
Longitude                   607
Month                         0
Reported by                   0
Falls within                  0
Location                      0
Crime type                    0
source_month                  0
dtype: int64

In [8]:
street["has_crime_id"] = street["Crime ID"].notna()
street.groupby("has_crime_id")["Crime type"].value_counts(normalize = True)

has_crime_id  Crime type                  
False         Anti-social behaviour           1.000000
True          Violence and sexual offences    0.459987
              Shoplifting                     0.117436
              Criminal damage and arson       0.101771
              Other theft                     0.070522
              Public order                    0.065962
              Burglary                        0.047151
              Vehicle crime                   0.037911
              Drugs                           0.034845
              Other crime                     0.024349
              Bicycle theft                   0.018850
              Possession of weapons           0.011077
              Robbery                         0.006742
              Theft from the person           0.003397
Name: proportion, dtype: float64

In [9]:
street["Crime type"].value_counts().head(10)

Crime type
Violence and sexual offences    34798
Anti-social behaviour           25456
Shoplifting                      8884
Criminal damage and arson        7699
Other theft                      5335
Public order                     4990
Burglary                         3567
Vehicle crime                    2868
Drugs                            2636
Other crime                      1842
Name: count, dtype: int64

In [10]:
street["Last outcome category"].value_counts().head(10)

Last outcome category
Unable to prosecute suspect                            31291
Investigation complete; no suspect identified          18850
Status update unavailable                               7134
Under investigation                                     5462
Court result unavailable                                4195
Awaiting court outcome                                  3379
Local resolution                                        1836
Further investigation is not in the public interest     1708
Action to be taken by another organisation               594
Further action is not in the public interest             421
Name: count, dtype: int64

In [11]:
street["Month"].value_counts().sort_index()

Month
2024-07    5704
2024-08    5535
2024-09    5114
2024-10    5393
2024-11    4760
2024-12    4606
2025-01    4459
2025-02    4334
2025-03    5212
2025-04    5408
2025-05    5599
2025-06    5449
2025-07    5885
2025-08    5580
2025-09    5119
2025-10    5098
2025-11    4876
2025-12    4504
2026-01    4255
2026-02    4216
Name: count, dtype: int64

# Outcomes EDA

In [12]:
print("Outcomes Shape:", outcomes.shape)

Outcomes Shape: (111474, 11)


In [38]:
outcomes.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Outcome type,source_month
0,7337fc0686f4a3caa58b8a322129fdffa6e09e8e961ee9...,2024-07-01,North Yorkshire Police,North Yorkshire Police,-1.275947,54.096398,On or near,E01027600,Hambleton 011A,Investigation complete; no suspect identified,2024-07
1,76bf63de8403d51e14722a6e0bcdd18d3d6cac508364ef...,2024-07-01,North Yorkshire Police,North Yorkshire Police,-1.073596,53.783210,On or near Gowthorpe,E01027908,Selby 005D,Suspect charged,2024-07
2,23269f9c365ddf04c4a5439207fa81314760af48f21966...,2024-07-01,North Yorkshire Police,North Yorkshire Police,NaN,NaN,No location,NaN,NaN,Investigation complete; no suspect identified,2024-07
3,0f61ac332836ad23e13bd97f4d844048c1b90ed7993114...,2024-07-01,North Yorkshire Police,North Yorkshire Police,-1.119030,53.951451,On or near Hamilton Drive West,E01013441,York 016E,Unable to prosecute suspect,2024-07
4,306ddc73d7bbe72548372bbc25062559ad3fa0d92d0f12...,2024-07-01,North Yorkshire Police,North Yorkshire Police,-0.849165,54.079651,On or near Kirkham View,E01027778,Ryedale 007B,Unable to prosecute suspect,2024-07


In [39]:
outcomes.isna().sum().sort_values(ascending = False)

LSOA name       305
LSOA code       305
Longitude       304
Latitude        304
Reported by       0
Crime ID          0
Month             0
Location          0
Falls within      0
Outcome type      0
source_month      0
dtype: int64

In [13]:
print("Outcomes Info:", outcomes.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111474 entries, 0 to 111473
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Crime ID      111474 non-null  object 
 1   Month         111474 non-null  object 
 2   Reported by   111474 non-null  object 
 3   Falls within  111474 non-null  object 
 4   Longitude     110967 non-null  float64
 5   Latitude      110967 non-null  float64
 6   Location      111474 non-null  object 
 7   LSOA code     110966 non-null  object 
 8   LSOA name     110966 non-null  object 
 9   Outcome type  111474 non-null  object 
 10  source_month  111474 non-null  object 
dtypes: float64(2), object(9)
memory usage: 9.4+ MB
Outcomes Info: None


In [14]:
outcomes["Crime ID"].value_counts().head()

Crime ID
ee67fa6a20a636defae089756f985241ec374eff9dd3ebfff8bbe7636eb2bc8d    12
2beda737d3c0b1fa233560007bfd9d5ea5cce1c99d409772ffe9b6dccbfbcffb    10
dd2b41466f3f1cc341b915cc748d5f69356130ba760463082d6d4b213b630c28    10
728eb975f3742e581bbc867c5d0c9610dc1927470633cf3b4fbff9a87122e1f6    10
ee6ce17d9d3f27cce4227cf5130a0fe3cf05726504ed177faee7f020560fbc5a    10
Name: count, dtype: int64

In [15]:
outcomes.groupby("Crime ID").size().describe()

count    73171.000000
mean         1.523472
std          0.702233
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         12.000000
dtype: float64

In [16]:
outcomes["Outcome type"].value_counts().head(10)

Outcome type
Unable to prosecute suspect                            59185
Investigation complete; no suspect identified          21896
Suspect charged                                        20462
Local resolution                                        3157
Further investigation is not in the public interest     2814
Offender given a caution                                 962
Formal action is not in the public interest              950
Action to be taken by another organisation               915
Further action is not in the public interest             881
Offender given a drugs possession warning                141
Name: count, dtype: int64

In [17]:
outcomes["Month"].value_counts().sort_index()

Month
2024-07    5853
2024-08    5815
2024-09    6312
2024-10    6406
2024-11    5790
2024-12    6578
2025-01    3803
2025-02    5228
2025-03    3152
2025-04    5996
2025-05    5195
2025-06    5667
2025-07    6198
2025-08    6361
2025-09    5864
2025-10    6401
2025-11    5564
2025-12    4972
2026-01    5325
2026-02    4994
Name: count, dtype: int64

# Search EDA

In [18]:
print("Search Shape:", search.shape)

Search Shape: (960, 16)


In [40]:
search.head()

,Type,Date,Part of a policing operation,Policing operation,Latitude,Longitude,Gender,Age range,Self-defined ethnicity,Officer-defined ethnicity,Legislation,Object of search,Outcome,Outcome linked to object of search,Removal of more than just outer clothing,source_month
0,Person and Vehicle search,2024-06-30T23:04:00+00:00,NaN,NaN,53.902243,-1.381316,Male,25-34,White - Any other White background,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,NaN,False,NaN,2024-07
1,Person search,2024-07-02T01:26:00+00:00,NaN,NaN,54.230027,-1.345899,Male,over 34,White - English/Welsh/Scottish/Northern Irish/...,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,NaN,False,NaN,2024-07
2,Person search,2024-07-02T19:50:00+00:00,NaN,NaN,53.957273,-1.084983,Male,over 34,White - Irish,NaN,Police and Criminal Evidence Act 1984 (section 1),Article for use in theft,NaN,False,NaN,2024-07
3,Person search,2024-07-02T22:18:00+00:00,NaN,NaN,53.961927,-1.090718,Male,over 34,White - English/Welsh/Scottish/Northern Irish/...,NaN,Police and Criminal Evidence Act 1984 (section 1),Controlled drugs,NaN,False,NaN,2024-07
4,Person search,2024-07-03T02:33:00+00:00,NaN,NaN,53.956569,-1.082240,Male,over 34,White - English/Welsh/Scottish/Northern Irish/...,White,Police and Criminal Evidence Act 1984 (section 1),Stolen goods,Arrest,True,NaN,2024-07


In [41]:
search.isna().sum().sort_values(ascending = False)

Policing operation                          960
Part of a policing operation                960
Removal of more than just outer clothing    960
Outcome                                     742
Officer-defined ethnicity                   187
Self-defined ethnicity                       84
Legislation                                  36
Object of search                             36
Latitude                                     29
Longitude                                    29
Outcome linked to object of search            8
Gender                                        6
Age range                                     5
Type                                          0
Date                                          0
source_month                                  0
dtype: int64

In [19]:
search.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 16 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Type                                      960 non-null    object 
 1   Date                                      960 non-null    object 
 2   Part of a policing operation              0 non-null      float64
 3   Policing operation                        0 non-null      float64
 4   Latitude                                  931 non-null    float64
 5   Longitude                                 931 non-null    float64
 6   Gender                                    954 non-null    object 
 7   Age range                                 955 non-null    object 
 8   Self-defined ethnicity                    876 non-null    object 
 9   Officer-defined ethnicity                 773 non-null    object 
 10  Legislation                           

In [20]:
search["Outcome"].value_counts()

Outcome
Arrest                          125
A no further action disposal     44
Community resolution             35
Khat or Cannabis warning         10
Summons / charged by post         3
Penalty Notice for Disorder       1
Name: count, dtype: int64

In [21]:
search["Object of search"].value_counts()

Object of search
Controlled drugs                       581
Stolen goods                           149
Offensive weapons                      111
Article for use in theft                67
Firearms                                 8
Articles for use in criminal damage      8
Name: count, dtype: int64

In [22]:
search["Gender"].value_counts()

Gender
Male      802
Female    152
Name: count, dtype: int64

In [23]:
search["Age range"].value_counts()

Age range
over 34     330
18-24       235
25-34       230
10-17       157
under 10      3
Name: count, dtype: int64

In [37]:
search["Outcome linked to object of search"]

0      False
1      False
2      False
3      False
4       True
       ...  
955    False
956    False
957      NaN
958    False
959    False
Name: Outcome linked to object of search, Length: 960, dtype: object

In [25]:
search["Date"].head()

0    2024-06-30T23:04:00+00:00
1    2024-07-02T01:26:00+00:00
2    2024-07-02T19:50:00+00:00
3    2024-07-02T22:18:00+00:00
4    2024-07-03T02:33:00+00:00
Name: Date, dtype: object

# Cross Table EDA

In [26]:
print("Street rows:", len(street))
print("Outcomes rows:", len(outcomes))

Street rows: 101106
Outcomes rows: 111474


In [27]:
with_id = street[street["Crime ID"].notna()]
without_id = street[street["Crime ID"].isna()]

with_id = with_id.drop_duplicates(subset = ["Crime ID"], keep = "first")
street = pd.concat([with_id, without_id], ignore_index = True)

print("Street rows:", len(street))

Street rows: 99837


In [28]:
outcomes = outcomes.drop_duplicates(
    subset = ["Crime ID", "Month", "Outcome type"],
    keep = "first",
    ignore_index = True
)

print("Outcomes rows:", len(outcomes))

Outcomes rows: 74662


In [29]:
outcomes["Month"] = pd.to_datetime(outcomes["Month"], errors = "coerce")

latest_outcomes = (
    outcomes
    .sort_values("Month")
    .drop_duplicates(subset = ["Crime ID"], keep = "last")
)

latest_outcomes = latest_outcomes.rename(columns = {
    "Month": "Latest outcome month",
    "Outcome type": "Latest outcome type"
})

print("Latest outcomes rows:", len(latest_outcomes))

Latest outcomes rows: 73171


In [30]:
joined = street.merge(
    latest_outcomes[["Crime ID", "Latest outcome month", "Latest outcome type"]],
    on = "Crime ID",
    how = "left",
    indicator = True
)

In [31]:
print("Joined rows:", len(joined))

Joined rows: 99837


In [32]:
joined["_merge"].value_counts()

_merge
both          63054
left_only     36783
right_only        0
Name: count, dtype: int64

In [33]:
left_only = joined[
    (joined["_merge"] == "left_only") &
    (joined["Crime ID"].notna())
]

left_only

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context,source_month,has_crime_id,Latest outcome month,Latest outcome type,_merge
14,a78cc6a761f81453e4a6993e6354f26a64389f6cf14e14...,2024-07,North Yorkshire Police,North Yorkshire Police,-2.468121,54.153805,On or near Supermarket,E01027570,Craven 001D,Drugs,Status update unavailable,NaN,2024-07,True,NaT,NaN,left_only
27,cf8b65c7a2f413202f49138f4d638b0134cea8091ecc77...,2024-07,North Yorkshire Police,North Yorkshire Police,-2.260198,54.021275,On or near Tranmere Court,E01027568,Craven 003A,Possession of weapons,Status update unavailable,NaN,2024-07,True,NaT,NaN,left_only
85,b77d440527a4c1c80b18168f865240b5f77e64d92a35ec...,2024-07,North Yorkshire Police,North Yorkshire Police,-2.031252,53.958720,On or near Back Midland Street,E01034956,Craven 006E,Public order,Status update unavailable,NaN,2024-07,True,NaT,NaN,left_only
89,250b14d51e9a5430fac01b03ae4dab4edbef06c2d65d82...,2024-07,North Yorkshire Police,North Yorkshire Police,-2.031252,53.958720,On or near Back Midland Street,E01034956,Craven 006E,Violence and sexual offences,Status update unavailable,NaN,2024-07,True,NaT,NaN,left_only
96,4503969a14224edd9ae71275d8ba605d1b1e25f7bb2cd3...,2024-07,North Yorkshire Police,North Yorkshire Police,-2.018718,53.958964,On or near Bus/Coach Station,E01034957,Craven 006F,Other theft,Status update unavailable,NaN,2024-07,True,NaT,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74376,9425f37e5f927ebca68304f4dd2826cdfe2095682ccce9...,2026-02,North Yorkshire Police,North Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Shoplifting,Under investigation,NaN,2026-02,True,NaT,NaN,left_only
74377,21e54268c1fa95ad1a3f949c82f9c3d618cc731ede6cb5...,2026-02,North Yorkshire Police,North Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Vehicle crime,Under investigation,NaN,2026-02,True,NaT,NaN,left_only
74378,4f25f33df03a7b14ce8eff9efa4e4a2fd666a79711d84c...,2026-02,North Yorkshire Police,North Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Under investigation,NaN,2026-02,True,NaT,NaN,left_only
74379,ee3b9bd56d5f91cd51581577c6214b4c6f7cecc9a5d97d...,2026-02,North Yorkshire Police,North Yorkshire Police,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Under investigation,NaN,2026-02,True,NaT,NaN,left_only


In [34]:
left_only["Crime type"].value_counts()

Crime type
Violence and sexual offences    6590
Criminal damage and arson        796
Public order                     679
Other theft                      597
Drugs                            546
Burglary                         545
Shoplifting                      518
Other crime                      429
Vehicle crime                    279
Possession of weapons            148
Robbery                           97
Bicycle theft                     86
Theft from the person             17
Name: count, dtype: int64

In [35]:
left_only["Last outcome category"].value_counts()

Last outcome category
Status update unavailable    6280
Under investigation          5047
Name: count, dtype: int64